# Chuẩn hóa dữ liệu bằng Min-Max

Phương pháp: **Min-Max scaling**

Notebook đọc dữ liệu từ `github_repos_complete.csv`, chuẩn hóa các cột số được chỉ định và lưu kết quả ra file riêng. File gốc không bị ghi đè.

Công thức: `x_scaled = (x - min) / (max - min)`. Giá trị nhỏ nhất thành 0 và lớn nhất thành 1 đối với mỗi cột có biến thiên.

**Các cột được chuẩn hóa:**
- `star_count`
- `forks_count`
- `open_issues_count`
- `size_kb`


## 1. Đọc dữ liệu và kiểm tra cột

In [2]:
from pathlib import Path
import pandas as pd
import numpy as np

INPUT_FILE = Path("github_repos_text_filled.csv")
OUTPUT_FILE = Path("github_repos_minmax_scaled.csv")

if not INPUT_FILE.exists():
    raise FileNotFoundError(
        f"Không tìm thấy file: {INPUT_FILE.resolve()}. "
        "Hãy đặt CSV cùng thư mục với notebook hoặc sửa INPUT_FILE."
    )

df = pd.read_csv(INPUT_FILE, low_memory=False)

TARGET_COLUMNS = ['star_count', 'forks_count', 'open_issues_count', 'size_kb']
missing_columns = set(TARGET_COLUMNS) - set(df.columns)
if missing_columns:
    raise ValueError(f"CSV thiếu các cột bắt buộc: {missing_columns}")

print(f"Số dòng: {len(df):,}")
print("Số giá trị thiếu ban đầu:")
display(df[TARGET_COLUMNS].isna().sum().to_frame("missing_count"))


Số dòng: 12,000
Số giá trị thiếu ban đầu:


,missing_count
star_count,0
forks_count,0
open_issues_count,0
size_kb,0


## 2. Chuẩn hóa các cột số

Chuyển các cột đích sang kiểu số. Giá trị không chuyển đổi được sẽ thành thiếu (`NaN`) và được giữ là thiếu khi chuẩn hóa.

In [3]:
# Ép kiểu số; giá trị không hợp lệ sẽ trở thành NaN
for column in TARGET_COLUMNS:
    df[column] = pd.to_numeric(df[column], errors="coerce")

before_stats = df[TARGET_COLUMNS].describe().T
display(before_stats)


,count,mean,std,min,25%,50%,75%,max
star_count,12000.0,4657.808833,13516.041706,612.0,1065.00,1713.0,3483.00,393057.0
forks_count,12000.0,601.179750,2518.165190,0.0,88.00,174.0,392.00,107748.0
open_issues_count,12000.0,79.025167,596.567969,0.0,5.00,18.0,53.00,51103.0
size_kb,12000.0,109556.913917,889179.717088,0.0,2165.75,13518.5,58916.25,57192250.0


## 3. Áp dụng Min-Max scaling

Công thức: `x_scaled = (x - min) / (max - min)`. Giá trị nhỏ nhất thành 0 và lớn nhất thành 1 đối với mỗi cột có biến thiên.



In [4]:
# Min-Max scaling: x' = (x - min) / (max - min)
for column in TARGET_COLUMNS:
    col_min = df[column].min()
    col_max = df[column].max()
    if pd.isna(col_min) or pd.isna(col_max):
        # Cột toàn giá trị thiếu: giữ nguyên
        continue
    if col_max == col_min:
        # Cột hằng số: gán 0 cho các giá trị không thiếu
        df[column] = df[column].where(df[column].isna(), 0.0)
    else:
        df[column] = (df[column] - col_min) / (col_max - col_min)

print("Min-Max hoàn tất. Với cột không hằng số, giá trị không thiếu thường nằm trong [0, 1].")

Min-Max hoàn tất. Với cột không hằng số, giá trị không thiếu thường nằm trong [0, 1].


## 4. Lưu file kết quả và kiểm tra

In [5]:
df.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")
print(f"Đã lưu file: {OUTPUT_FILE.resolve()}")

print("Thống kê sau chuẩn hóa:")
display(df[TARGET_COLUMNS].describe().T)

print("Số giá trị thiếu sau chuẩn hóa:")
display(df[TARGET_COLUMNS].isna().sum().to_frame("missing_count"))

display(df[TARGET_COLUMNS].head(10))


Đã lưu file: D:\HomeWork\IE313\Đồ án\analysis-popularity-github-repository\notebooks\chuan_hoa\github_repos_minmax_scaled.csv
Thống kê sau chuẩn hóa:


,count,mean,std,min,25%,50%,75%,max
star_count,12000.0,0.010309,0.034441,0.0,0.001154,0.002805,0.007316,1.0
forks_count,12000.0,0.005579,0.023371,0.0,0.000817,0.001615,0.003638,1.0
open_issues_count,12000.0,0.001546,0.011674,0.0,0.000098,0.000352,0.001037,1.0
size_kb,12000.0,0.001916,0.015547,0.0,0.000038,0.000236,0.001030,1.0


Số giá trị thiếu sau chuẩn hóa:


,missing_count
star_count,0
forks_count,0
open_issues_count,0
size_kb,0


,star_count,forks_count,open_issues_count,size_kb
0,0.397877,0.211855,0.003424,0.008974
1,0.230761,0.034312,0.058842,0.003922
2,0.151580,0.082424,0.017690,0.005286
3,0.107579,0.082619,0.000470,0.000041
4,0.101999,0.027323,0.009647,0.000367
5,0.083556,0.026924,0.000489,0.000581
6,0.075188,0.026868,0.003053,0.002455
7,0.062791,0.024678,0.000000,0.005135
8,0.061349,0.007917,0.003464,0.000020
9,0.060388,0.027267,0.003464,0.000222


## Lưu ý
- Chỉ bốn cột nêu trên được biến đổi; các cột khác được giữ nguyên.
- Giá trị thiếu không được tự động điền.
- Các phép chuẩn hóa được fit trên toàn bộ file đầu vào. Nếu dùng dữ liệu cho mô hình học máy, hãy fit bộ chuẩn hóa chỉ trên tập train rồi áp dụng cho tập test để tránh data leakage.
